# testing.ipynb — Wildfire Smoke Detection (YOLOv8m)

Downloads the D-Fire dataset, optionally retrains the improved model, then
runs the full evaluation battery on the held-out test split.

Usage:
    1. Runtime > Change runtime type > T4 GPU
    2. Upload baseline_best.pt and improved_best.pt to
       /MyDrive/wildfire-weights/
    3. Add KAGGLE_USERNAME and KAGGLE_KEY to Colab Secrets
    4. Run top to bottom

Dataset: sayedgamal99/smoke-fire-detection-yolo (D-Fire mirror).
Split: stratified 80/10/10 with SEED=42, cached to Drive on first run.

Sections:
    0. Setup
    1. Paths
    2. Dataset (Kaggle CLI download + split)
    3. Retrain improved (optional, ~45 min)
    4. Helpers — parse yolo val logs
    5. Core evaluation (yolo val)
    6. Threshold sweep
    7. Robustness to perturbations
    8. Speed benchmark (yolo benchmark)
    9. Qualitative samples (yolo predict)
    10. Summary

## 0 · Setup

In [ ]:
# 0.1. Check T4
!nvidia-smi -L


In [ ]:
# 0.2. Install
!pip install -q ultralytics albumentations pandas seaborn


In [ ]:
# 0.3. Imports
import os, re, shutil, json, time
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import cv2
import albumentations as A


In [ ]:
# 0.4. Mount Drive
from google.colab import drive
drive.mount("/content/drive")


## 1 · Paths

In [ ]:
# 1.1. All paths defined once. Edit if your Drive layout differs.
DRIVE        = Path("/content/drive/MyDrive")
WEIGHTS_DIR  = DRIVE / "wildfire-weights"
DATA_DIR     = DRIVE / "wildfire-data" / "processed"

BASELINE_PT  = WEIGHTS_DIR / "baseline_best.pt"
IMPROVED_PT  = WEIGHTS_DIR / "improved_best.pt"

TEST_IMAGES  = DATA_DIR / "images" / "test"
TEST_LABELS  = DATA_DIR / "labels" / "test"
RESULTS_DIR  = WEIGHTS_DIR / "test_results"
LOG_DIR      = Path("/content/logs"); LOG_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("weights :", WEIGHTS_DIR)
print("data    :", DATA_DIR)
print("results :", RESULTS_DIR)
print("logs    :", LOG_DIR)


In [ ]:
# 1.2. Check .pt files exist
missing = []
for label, p in [("baseline weights", BASELINE_PT),
                 ("improved weights", IMPROVED_PT)]:
    if not p.exists():
        missing.append((label, p))

if missing:
    print("=" * 70)
    print("MISSING:")
    for label, p in missing:
        print(f"  [{label}]  {p}")
    print("=" * 70)
    raise FileNotFoundError(f"{len(missing)} path(s) missing")

print("Weights OK.")


In [ ]:
# 1.3. Copy .pt files to local disk so CLI reads don't hit Drive mid-run
LOCAL_CKPT = Path("/content/checkpoints"); LOCAL_CKPT.mkdir(exist_ok=True)
for src in (BASELINE_PT, IMPROVED_PT):
    dst = LOCAL_CKPT / src.name
    if not dst.exists():
        shutil.copy(src, dst)
BASELINE_PT = LOCAL_CKPT / "baseline_best.pt"
IMPROVED_PT = LOCAL_CKPT / "improved_best.pt"
print("local:", BASELINE_PT, IMPROVED_PT)


## 2 · Dataset — `kaggle datasets download`

First run: downloads D-Fire from Kaggle, splits 80/10/10 with `SEED=42`,
caches the processed folder to Drive.
Every run after: reuses the cached copy.

In [ ]:
# 2.1. Kaggle credentials
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")
print("kaggle user:", os.environ["KAGGLE_USERNAME"])


In [ ]:
# 2.2. Download + split if DATA_DIR isn't already on Drive
import random, subprocess

RAW_DIR = Path("/content/raw"); RAW_DIR.mkdir(exist_ok=True)
LOCAL_PROCESSED = Path("/content/processed")
REMAP = {0: 1, 1: 0}   # D-Fire raw: 0=smoke → 1,  1=fire → 0
SEED  = 42

def _remap(src, dst):
    lines = src.read_text().strip().splitlines() if src.stat().st_size else []
    out = []
    for line in lines:
        parts = line.split()
        if len(parts) == 5:
            parts[0] = str(REMAP.get(int(parts[0]), int(parts[0])))
        out.append(" ".join(parts))
    dst.write_text("\n".join(out) + ("\n" if out else ""))

def _classes(lbl):
    if not lbl.exists() or lbl.stat().st_size == 0: return set()
    return {REMAP.get(int(l.split()[0]), int(l.split()[0]))
            for l in lbl.read_text().strip().splitlines() if l.split()}

def _bucket(cs):
    if 0 in cs and 1 in cs: return "both"
    if 0 in cs: return "fire"
    if 1 in cs: return "smoke"
    return "empty"

if DATA_DIR.exists() and (DATA_DIR / "images" / "train").exists():
    print(f">>> Using cached dataset at {DATA_DIR}")
else:
    print(">>> First-time setup")
    # 1) Kaggle CLI download
    !kaggle datasets download -d sayedgamal99/smoke-fire-detection-yolo -p /content
    !unzip -q /content/smoke-fire-detection-yolo.zip -d {RAW_DIR}
    !rm -f /content/smoke-fire-detection-yolo.zip

    # 2) Find all (image, label) pairs, ignoring any pre-existing split folders
    pairs = []
    for img in RAW_DIR.rglob("*"):
        if img.suffix.lower() not in {".jpg",".jpeg",".png"}: continue
        for cand in [img.parent.parent / "labels" / f"{img.stem}.txt",
                     img.parent / f"{img.stem}.txt",
                     img.with_suffix(".txt")]:
            if cand.exists():
                pairs.append((img, cand)); break
    print(f"  {len(pairs)} image/label pairs")
    assert pairs, f"No pairs found in {RAW_DIR}"

    # 3) Stratified 80/10/10 split
    buckets = {"fire": [], "smoke": [], "both": [], "empty": []}
    for img, lbl in pairs:
        buckets[_bucket(_classes(lbl))].append((img, lbl))
    rng = random.Random(SEED)
    train, val, test = [], [], []
    for items in buckets.values():
        rng.shuffle(items)
        n = len(items); n_tr = int(n * 0.8); n_va = (n - n_tr) // 2
        train += items[:n_tr]; val += items[n_tr:n_tr+n_va]; test += items[n_tr+n_va:]
    print(f"  split → train {len(train)} | val {len(val)} | test {len(test)}")

    # 4) Copy to LOCAL_PROCESSED with remapped labels
    for split, items in [("train", train), ("val", val), ("test", test)]:
        (LOCAL_PROCESSED / "images" / split).mkdir(parents=True, exist_ok=True)
        (LOCAL_PROCESSED / "labels" / split).mkdir(parents=True, exist_ok=True)
        for img, lbl in items:
            shutil.copy2(img, LOCAL_PROCESSED / "images" / split / img.name)
            _remap(lbl, LOCAL_PROCESSED / "labels" / split / f"{img.stem}.txt")

    # 5) Cache to Drive
    print(">>> Caching to Drive…")
    shutil.copytree(LOCAL_PROCESSED, str(DATA_DIR), dirs_exist_ok=True)
    print(">>> Done.")

for s in ("train","val","test"):
    n = len(list((DATA_DIR / "images" / s).glob("*"))) if (DATA_DIR/"images"/s).exists() else 0
    print(f"  {s:>5}: {n} images")


In [ ]:
# 2.3. Absolute-path dataset.yaml for `yolo val`
DATA_YAML = Path("/content/dataset.yaml")
DATA_YAML.write_text(yaml.dump({
    "path":  str(DATA_DIR),
    "train": "images/train",
    "val":   "images/val",
    "test":  "images/test",
    "nc":    2,
    "names": ["fire", "smoke"],
}))
!cat {DATA_YAML}


## 3 · Retrain improved model (optional — skip to §4 if you just want testing)

The original `improved_best.pt` underperformed baseline because it was
**undertrained**: 15 epochs vs baseline's 25, 10× lower LR, cold-started from
yolov8m.pt. This section fixes all three:

- **Warm-starts from `baseline_best.pt`** (not yolov8m.pt) → can only improve on baseline
- **40 epochs at 832px** with SGD + cos_lr + `patience=10`
- Custom Albumentations pipeline (fog, motion blur, noise, shadow) for wildfire robustness

Training takes ~45–60 min on T4. On completion, the new weights overwrite
`/MyDrive/wildfire-weights/improved_best.pt` and the old file is renamed to
`improved_best_v1_backup.pt`.

Skip to section 4 if you want to evaluate the existing `.pt` files as-is.

In [ ]:
# 3.1. Install custom Albumentations pipeline
# Ultralytics calls this with just `image=...` — bbox augmentation is handled separately
# in its own pipeline, so bbox_params here would crash with 'label_fields not valid'.
def install_wildfire_pipeline():
    from ultralytics.data import augment as _aug
    class WildfireAug(_aug.Albumentations):
        def __init__(self, p=1.0, **kwargs):
            self.p = p
            self.transform = A.Compose([
                A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
                A.RandomFog(fog_coef_lower=0.05, fog_coef_upper=0.30, alpha_coef=0.08, p=0.25),
                A.MotionBlur(blur_limit=5, p=0.20),
                A.GaussNoise(var_limit=(10.0, 40.0), p=0.20),
                A.RandomShadow(p=0.20),
                A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=20,
                                     val_shift_limit=15, p=0.40),
            ])  # no bbox_params — these are image-only color/noise transforms
            self.contains_spatial = False
    _aug.Albumentations = WildfireAug
    print("WildfireAug pipeline installed")


In [ ]:
# 3.2. Training config
TRAIN_CFG = {
    "data":     str(DATA_YAML),
    "project":  "/content/training_output",
    "name":     "improved_v2",
    "epochs":   40,
    "batch":    16,
    "imgsz":    832,
    "device":   0,
    "workers":  4,
    "cache":    False,
    "optimizer":       "SGD",
    "lr0":             0.005,
    "lrf":             0.01,
    "momentum":        0.937,
    "weight_decay":    0.0005,
    "cos_lr":          True,
    "warmup_epochs":   3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr":  0.1,
    "box":        7.5,
    "cls":        1.0,
    "dfl":        1.5,
    "hsv_h":      0.015, "hsv_s":  0.7, "hsv_v": 0.4,
    "degrees":    5.0,   "translate": 0.1, "scale": 0.5, "shear": 2.0,
    "fliplr":     0.5,
    "mosaic":     1.0,   "mixup":  0.10, "copy_paste": 0.20,
    "patience":   10,
}
print("epochs:", TRAIN_CFG["epochs"], "| imgsz:", TRAIN_CFG["imgsz"],
      "| lr0:", TRAIN_CFG["lr0"], "| warm start:", BASELINE_PT.name)


In [ ]:
# 3.3. Train. Takes ~45 min on T4.
from ultralytics import YOLO

install_wildfire_pipeline()

model = YOLO(str(BASELINE_PT))
t0 = time.time()
results = model.train(**TRAIN_CFG)
print(f"\ntraining took {(time.time()-t0)/60:.1f} min")
print(f"final mAP@50:    {results.box.map50:.4f}")
print(f"final mAP@50-95: {results.box.map:.4f}")


In [ ]:
# 3.4. Find new best.pt, back up old improved_best.pt, save new one to Drive
import glob

candidates = [
    Path("/content/training_output/improved_v2/weights/best.pt"),
    Path("runs/detect/training_output/improved_v2/weights/best.pt"),
]
candidates += [Path(p) for p in glob.glob("/content/**/improved_v2/weights/best.pt", recursive=True)]
new_best = next((p for p in candidates if p.exists()), None)
assert new_best, "could not locate new best.pt"
print("found new weights:", new_best)

drive_improved = WEIGHTS_DIR / "improved_best.pt"
backup         = WEIGHTS_DIR / "improved_best_v1_backup.pt"
if drive_improved.exists() and not backup.exists():
    shutil.copy(drive_improved, backup)
    print(f"backed up old → {backup.name}")

shutil.copy(new_best, drive_improved)
shutil.copy(new_best, IMPROVED_PT)   # refresh local copy for §4+ below
print(f"saved to Drive: {drive_improved}")
print(f"refreshed local: {IMPROVED_PT}")


## 4 · Helpers — parse metrics from `yolo val` logs

`yolo val` prints a metrics block like:
```
         Class   Images  Instances     P       R    mAP50  mAP50-95
           all      100        234   0.823   0.791   0.854   0.612
          fire      100        145   0.850   0.820   0.880   0.640
         smoke      100         89   0.795   0.762   0.828   0.584
```
Two helpers: one for the `all` summary row, one for per-class rows.

In [ ]:
# 4.1. Parsers
CLASS_NAMES = ["fire", "smoke"]

def _parse_row(log_text: str, class_name: str):
    '''Grab (P, R, mAP50, mAP50-95) from a single row of a yolo val log.'''
    pattern = rf'^\s*{class_name}\s+\d+\s+\d+\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)'
    m = re.search(pattern, log_text, re.MULTILINE)
    if not m:
        return {"precision": 0.0, "recall": 0.0, "mAP50": 0.0, "mAP50-95": 0.0}
    p, r, m50, m95 = map(float, m.groups())
    return {"precision": p, "recall": r, "mAP50": m50, "mAP50-95": m95}

def parse_overall(log_path):
    '''Return metrics from the `all` row.'''
    return _parse_row(Path(log_path).read_text(), "all")

def parse_per_class(log_path):
    '''Return {class_name: metrics} for fire and smoke.'''
    text = Path(log_path).read_text()
    return {c: _parse_row(text, c) for c in CLASS_NAMES}

def f1_score(p, r):
    return 2 * p * r / (p + r + 1e-9)


## 5 · Core evaluation — `yolo val`

In [ ]:
# 5.1. Evaluate baseline
!yolo val model={BASELINE_PT} data={DATA_YAML} split=test imgsz=640 batch=16 \
         device=0 plots=True save_json=True \
         project=runs/eval name=baseline 2>&1 | tee {LOG_DIR}/baseline_eval.log


In [ ]:
# 5.2. Evaluate improved
!yolo val model={IMPROVED_PT} data={DATA_YAML} split=test imgsz=640 batch=16 \
         device=0 plots=True save_json=True \
         project=runs/eval name=improved 2>&1 | tee {LOG_DIR}/improved_eval.log


In [ ]:
# 5.3. Overall metrics table
overall_rows = []
for model_name in ("baseline", "improved"):
    metrics = parse_overall(LOG_DIR / f"{model_name}_eval.log")
    metrics["F1"] = f1_score(metrics["precision"], metrics["recall"])
    overall_rows.append({"model": model_name, **{k: round(v, 4) for k, v in metrics.items()}})

df_overall = pd.DataFrame(overall_rows)
df_overall.to_csv(RESULTS_DIR / "overall_metrics.csv", index=False)
df_overall


In [ ]:
# 5.4. Per-class metrics
per_class_rows = []
for model_name in ("baseline", "improved"):
    per_class = parse_per_class(LOG_DIR / f"{model_name}_eval.log")
    for class_name, metrics in per_class.items():
        metrics["F1"] = f1_score(metrics["precision"], metrics["recall"])
        per_class_rows.append({
            "model": model_name,
            "class": class_name,
            **{k: round(v, 4) for k, v in metrics.items()},
        })

df_per_class = pd.DataFrame(per_class_rows)
df_per_class.to_csv(RESULTS_DIR / "per_class_metrics.csv", index=False)
df_per_class.pivot(index="class", columns="model",
                   values=["precision","recall","F1","mAP50","mAP50-95"])


In [ ]:
# 5.5. Per-class bar chart
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, metric in zip(axes, ("mAP50", "F1")):
    pivot = df_per_class.pivot(index="class", columns="model", values=metric)
    pivot.plot(kind="bar", ax=ax, rot=0)
    ax.set_title(f"{metric} by class")
    ax.set_ylim(0, 1); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# 5.6. Show Ultralytics-generated plots
from IPython.display import Image, display
for plot_name in ("confusion_matrix.png", "PR_curve.png", "F1_curve.png"):
    print(f"=== {plot_name} ===")
    for model_name in ("baseline", "improved"):
        plot_path = Path(f"runs/eval/{model_name}/{plot_name}")
        if plot_path.exists():
            print(model_name); display(Image(str(plot_path), width=480))


## 6 · Threshold sweep

In [ ]:
# 6.1. Sweep conf × IoU, capture a log per combination
CONF_GRID = [0.25, 0.35, 0.45]
IOU_GRID  = [0.45, 0.55, 0.65]

sweep_rows = []
best_thresholds = {}

for model_name, weights_path in [("baseline", BASELINE_PT), ("improved", IMPROVED_PT)]:
    best = {"F1": -1}
    for conf in CONF_GRID:
        for iou in IOU_GRID:
            run_name = f"{model_name}_c{conf}_i{iou}"
            log_path = LOG_DIR / f"sweep_{run_name}.log"
            !yolo val model={weights_path} data={DATA_YAML} split=test imgsz=640 batch=16 \
                device=0 conf={conf} iou={iou} plots=False verbose=False \
                project=runs/sweep name={run_name} > {log_path} 2>&1

            metrics = parse_overall(log_path)
            score   = f1_score(metrics["precision"], metrics["recall"])
            sweep_rows.append({"model": model_name, "conf": conf, "iou": iou,
                               "precision": round(metrics["precision"], 4),
                               "recall":    round(metrics["recall"], 4),
                               "F1":        round(score, 4)})
            if score > best["F1"]:
                best = {"conf": conf, "iou": iou, "F1": round(score, 4)}
    best_thresholds[model_name] = best
    print(f"best {model_name}: {best}")

df_sweep = pd.DataFrame(sweep_rows)
df_sweep.to_csv(RESULTS_DIR / "threshold_sweep.csv", index=False)
with open(RESULTS_DIR / "best_thresholds.json", "w") as f:
    json.dump(best_thresholds, f, indent=2)


In [ ]:
# 6.2. Heatmap
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, model_name in zip(axes, ("baseline", "improved")):
    pivot = df_sweep[df_sweep.model == model_name].pivot(index="conf", columns="iou", values="F1")
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="viridis", ax=ax)
    ax.set_title(f"F1 — {model_name}")
plt.tight_layout(); plt.show()


## 7 · Robustness to perturbations

In [ ]:
# 7.1. Build perturbed test sets
PERTURBATIONS = {
    "clean":    None,
    "bright":   A.RandomBrightnessContrast(brightness_limit=(0.4, 0.4),
                                           contrast_limit=0, p=1.0),
    "dark":     A.RandomBrightnessContrast(brightness_limit=(-0.4, -0.4),
                                           contrast_limit=0, p=1.0),
    "blur":     A.GaussianBlur(blur_limit=(7, 7), p=1.0),
    "noise":    A.GaussNoise(var_limit=(40.0, 80.0), p=1.0),
    "fog":      A.RandomFog(fog_coef_lower=0.3, fog_coef_upper=0.5, p=1.0),
}

PERTURB_ROOT = Path("/content/perturbed"); PERTURB_ROOT.mkdir(exist_ok=True)
perturb_yamls = {}

for perturb_name, transform in PERTURBATIONS.items():
    base_dir = PERTURB_ROOT / perturb_name
    (base_dir / "images/test").mkdir(parents=True, exist_ok=True)
    (base_dir / "labels/test").mkdir(parents=True, exist_ok=True)

    for img_path in list(TEST_IMAGES.glob("*")):
        img = cv2.imread(str(img_path))
        if img is None: continue
        out = transform(image=img)["image"] if transform else img
        cv2.imwrite(str(base_dir / "images/test" / img_path.name), out)
        label_path = TEST_LABELS / f"{img_path.stem}.txt"
        if label_path.exists():
            shutil.copy2(label_path, base_dir / "labels/test" / label_path.name)

    yaml_path = base_dir / "dataset.yaml"
    yaml_path.write_text(yaml.dump({
        "path":  str(base_dir),
        "train": "images/test", "val": "images/test", "test": "images/test",
        "nc":    2, "names": ["fire", "smoke"],
    }))
    perturb_yamls[perturb_name] = str(yaml_path)

print("built:", list(perturb_yamls))


In [ ]:
# 7.2. Evaluate each model on each perturbation
robust_rows = []
for model_name, weights_path in [("baseline", BASELINE_PT), ("improved", IMPROVED_PT)]:
    for perturb_name, perturb_yaml in perturb_yamls.items():
        run_name = f"{model_name}_{perturb_name}"
        log_path = LOG_DIR / f"robust_{run_name}.log"
        !yolo val model={weights_path} data={perturb_yaml} split=test imgsz=640 batch=16 \
            device=0 plots=False verbose=False \
            project=runs/robust name={run_name} > {log_path} 2>&1

        metrics = parse_overall(log_path)
        robust_rows.append({"model": model_name, "perturbation": perturb_name,
                            "mAP50": round(metrics["mAP50"], 4),
                            "F1":    round(f1_score(metrics["precision"], metrics["recall"]), 4)})

df_robust = pd.DataFrame(robust_rows)
df_robust.to_csv(RESULTS_DIR / "robustness.csv", index=False)
df_robust.pivot(index="perturbation", columns="model", values="mAP50")


In [ ]:
# 7.3. Bar chart
fig, ax = plt.subplots(figsize=(9, 4))
labels = list(PERTURBATIONS); x = np.arange(len(labels)); bar_width = 0.35
for i, model_name in enumerate(("baseline", "improved")):
    sub = df_robust[df_robust.model == model_name].set_index("perturbation").loc[labels]
    ax.bar(x + (i - 0.5) * bar_width, sub["mAP50"], bar_width, label=model_name)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("mAP@50"); ax.set_title("Robustness across perturbations")
ax.legend(); plt.tight_layout(); plt.show()


## 8 · Speed benchmark — `yolo benchmark`

In [ ]:
# 8.1. Benchmark baseline
!yolo benchmark model={BASELINE_PT} data={DATA_YAML} imgsz=640 device=0


In [ ]:
# 8.2. Benchmark improved
!yolo benchmark model={IMPROVED_PT} data={DATA_YAML} imgsz=640 device=0


## 9 · Qualitative samples — `yolo predict`

In [ ]:
# 9.1. Copy a handful of test images into a working folder
sample_dir = Path("/content/sample_test")
if sample_dir.exists(): shutil.rmtree(sample_dir)
sample_dir.mkdir()
for f in sorted(TEST_IMAGES.glob("*"))[:6]:
    shutil.copy(f, sample_dir / f.name)
!ls {sample_dir}


In [ ]:
# 9.2. Predict with baseline
baseline_conf = best_thresholds["baseline"]["conf"]
baseline_iou  = best_thresholds["baseline"]["iou"]
!yolo predict model={BASELINE_PT} source={sample_dir} imgsz=640 device=0 \
              conf={baseline_conf} iou={baseline_iou} save=True \
              project=runs/predict name=baseline


In [ ]:
# 9.3. Predict with improved
improved_conf = best_thresholds["improved"]["conf"]
improved_iou  = best_thresholds["improved"]["iou"]
!yolo predict model={IMPROVED_PT} source={sample_dir} imgsz=640 device=0 \
              conf={improved_conf} iou={improved_iou} save=True \
              project=runs/predict name=improved


In [ ]:
# 9.4. Show predictions side-by-side
for img_name in sorted(os.listdir(sample_dir)):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, model_name in zip(axes, ("baseline", "improved")):
        pred_path = Path(f"runs/predict/{model_name}/{img_name}")
        if pred_path.exists():
            ax.imshow(cv2.cvtColor(cv2.imread(str(pred_path)), cv2.COLOR_BGR2RGB))
            ax.set_title(f"{model_name} — {img_name}"); ax.axis("off")
    plt.tight_layout(); plt.show()


## 10 · Summary

In [ ]:
# 10.1. Verdict + copy plot folders to Drive
baseline_metrics = df_overall[df_overall.model == "baseline"].iloc[0]
improved_metrics = df_overall[df_overall.model == "improved"].iloc[0]
print(f"baseline  →  mAP50 {baseline_metrics['mAP50']:.3f}  |  F1 {baseline_metrics['F1']:.3f}")
print(f"improved  →  mAP50 {improved_metrics['mAP50']:.3f}  |  F1 {improved_metrics['F1']:.3f}")
print(f"Δ mAP50 = {improved_metrics['mAP50'] - baseline_metrics['mAP50']:+.3f}   "
      f"Δ F1 = {improved_metrics['F1'] - baseline_metrics['F1']:+.3f}")

for model_name in ("baseline", "improved"):
    src = Path(f"runs/eval/{model_name}")
    dst = RESULTS_DIR / f"eval_{model_name}"
    if src.exists():
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(src, dst)

print(f"\nAll saved to: {RESULTS_DIR}")
!ls {RESULTS_DIR}
